In [2]:
import os
import re
import glob

def parse_dma_file(file_path):
    """
    解析单个txt文件，提取start和end的DMA值
    返回间隔列表
    """
    intervals = []
    start_value = None
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                line = line.strip()
                if not line:
                    continue
                
                # 使用正则表达式匹配start和end行
                start_match = re.search(r'fthread\s+\d+\s+start\s+dma\s+(\d+)', line)
                end_match = re.search(r'fthread\s+\d+\s+end\s+dma\s+(\d+)', line)
                
                if start_match:
                    start_value = int(start_match.group(1))
                elif end_match and start_value is not None:
                    end_value = int(end_match.group(1))
                    interval = end_value - start_value
                    intervals.append(interval)
                    start_value = None  # 重置，等待下一个start
    
    except Exception as e:
        print(f"读取文件 {file_path} 时出错: {e}")
        return []
    
    return intervals

def calculate_average_interval(intervals):
    """
    计算间隔的平均值
    """
    if not intervals:
        return 0
    return sum(intervals) / len(intervals)

def analyze_multiple_files(directory_path=".", file_pattern="*.txt"):
    """
    分析指定目录下的所有txt文件
    """
    # 获取所有匹配的txt文件
    file_pattern_path = os.path.join(directory_path, file_pattern)
    txt_files = glob.glob(file_pattern_path)
    
    if not txt_files:
        print(f"在目录 {directory_path} 中没有找到匹配的txt文件")
        return
    
    print(f"找到 {len(txt_files)} 个txt文件")
    print("-" * 60)
    
    all_results = {}
    
    for file_path in sorted(txt_files):
        print(f"\n正在处理文件: {os.path.basename(file_path)}")
        
        # 解析文件获取间隔
        intervals = parse_dma_file(file_path)
        
        if intervals:
            # 计算平均值
            avg_interval = calculate_average_interval(intervals)
            all_results[file_path] = {
                'intervals': intervals,
                'average': avg_interval,
                'count': len(intervals)
            }
            
            print(f"  找到 {len(intervals)} 个间隔")
            print(f"  间隔值: {intervals}")
            print(f"  平均间隔: {avg_interval:.2f}")
        else:
            print(f"  未找到有效的start-end对")
            all_results[file_path] = {
                'intervals': [],
                'average': 0,
                'count': 0
            }
    
    # 打印汇总结果
    print("\n" + "="*60)
    print("汇总结果:")
    print("="*60)
    
    for file_path, result in all_results.items():
        filename = os.path.basename(file_path)
        print(f"{filename:30} | 间隔数: {result['count']:3} | 平均值: {result['average']:8.2f}")
    
    # 计算总体平均值
    all_intervals = []
    for result in all_results.values():
        all_intervals.extend(result['intervals'])
    
    if all_intervals:
        total_avg = calculate_average_interval(all_intervals)
        print("-" * 60)
        print(f"{'所有文件总体平均间隔:':30} | 间隔数: {len(all_intervals):3} | 平均值: {total_avg:8.2f}")
    
    return all_results

def analyze_single_file(file_path):
    """
    分析单个文件
    """
    print(f"正在分析文件: {file_path}")
    
    if not os.path.exists(file_path):
        print(f"文件不存在: {file_path}")
        return None
    
    intervals = parse_dma_file(file_path)
    
    if intervals:
        avg_interval = calculate_average_interval(intervals)
        print(f"找到 {len(intervals)} 个间隔")
        print(f"间隔值: {intervals}")
        print(f"平均间隔: {avg_interval:.2f}")
        return {
            'intervals': intervals,
            'average': avg_interval,
            'count': len(intervals)
        }
    else:
        print("未找到有效的start-end对")
        return None

if __name__ == "__main__":
    # 使用示例：
    
    # 方式1: 分析当前目录下的所有txt文件
    print("=== 分析当前目录下的所有txt文件 ===")
    analyze_multiple_files()
    
    # 方式2: 分析指定目录下的txt文件
    # analyze_multiple_files("./data", "*.txt")
    
    # 方式3: 分析单个文件
    # analyze_single_file("example.txt")
    
    # 方式4: 分析特定模式的文件
    # analyze_multiple_files(".", "dma_*.txt")

=== 分析当前目录下的所有txt文件 ===
找到 4 个txt文件
------------------------------------------------------------

正在处理文件: mmu.txt
  找到 6 个间隔
  间隔值: [213, 182, 166, 155, 150, 144]
  平均间隔: 168.33

正在处理文件: normal.txt
  找到 5 个间隔
  间隔值: [1682, 1682, 1682, 1682, 1682]
  平均间隔: 1682.00

正在处理文件: page_falut.txt
  找到 5 个间隔
  间隔值: [3439, 2849, 2879, 2946, 2967]
  平均间隔: 3016.00

正在处理文件: upim.txt
  找到 5 个间隔
  间隔值: [178, 159, 149, 149, 149]
  平均间隔: 156.80

汇总结果:
mmu.txt                        | 间隔数:   6 | 平均值:   168.33
normal.txt                     | 间隔数:   5 | 平均值:  1682.00
page_falut.txt                 | 间隔数:   5 | 平均值:  3016.00
upim.txt                       | 间隔数:   5 | 平均值:   156.80
------------------------------------------------------------
所有文件总体平均间隔:                    | 间隔数:  21 | 平均值:  1204.00
